# Huấn luyện Plant Disease AI — MobileNetV2 (15 lớp)

Notebook chạy trên Google Colab: kết nối Google Drive, chọn thư mục PlantVillage, chia dữ liệu 70/15/15 có phân tầng, huấn luyện Transfer Learning, fine-tuning, đánh giá trên tập test độc lập và lưu model cùng báo cáo.

**Chuẩn bị:** tải bộ PlantVillage gồm đúng 15 thư mục lớp lên Google Drive. Mặc định đặt tại MyDrive/PlantVillage hoặc sửa DATASET_DIR ở bước cấu hình. Không tải dataset lên GitHub.

Model mới được lưu riêng trong Drive; không ghi đè model hiện tại.

In [ ]:
# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone mã nguồn và cài thư viện bổ sung
import os, subprocess, sys
REPO_DIR = '/content/Plant-Disease-Detection'
if not os.path.isdir(REPO_DIR):
    subprocess.run(['git','clone','--branch','feature/reproducible-mobilenetv2-colab-training','https://github.com/ShimeKano/Plant-Disease-Detection.git',REPO_DIR],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','scikit-learn','matplotlib'],check=True)
print('Source ready:',REPO_DIR)

In [ ]:
# 3. Cấu hình dataset và tham số huấn luyện
DATASET_DIR = '/content/drive/MyDrive/PlantVillage'  # sửa nếu dataset ở vị trí khác
OUTPUT_DIR = '/content/plant_disease_training_output'
IMG_SIZE = 128
BATCH_SIZE = 32
HEAD_EPOCHS = 12
FINE_TUNE_EPOCHS = 8
SEED = 42
assert os.path.isdir(DATASET_DIR), f'Không tìm thấy dataset: {DATASET_DIR}'
print('Dataset:',DATASET_DIR,'| Output:',OUTPUT_DIR)

In [ ]:
# 4. Kiểm tra TensorFlow và GPU
import tensorflow as tf
print('TensorFlow:',tf.__version__)
gpus=tf.config.list_physical_devices('GPU')
print('GPU:',gpus if gpus else 'Không phát hiện GPU — có thể chạy CPU nhưng sẽ chậm hơn.')
for gpu in gpus:
    try: tf.config.experimental.set_memory_growth(gpu,True)
    except RuntimeError: pass

In [ ]:
# 5. Chạy pipeline huấn luyện và đánh giá
command=[sys.executable,f'{REPO_DIR}/src/train.py','--data-dir',DATASET_DIR,'--output-dir',OUTPUT_DIR,'--img-size',str(IMG_SIZE),'--batch-size',str(BATCH_SIZE),'--epochs',str(HEAD_EPOCHS),'--fine-tune-epochs',str(FINE_TUNE_EPOCHS),'--seed',str(SEED)]
print('Running:', ' '.join(command))
subprocess.run(command,check=True)

In [ ]:
# 6. Đóng gói và lưu artifacts vào Google Drive
import shutil
from pathlib import Path
out=Path(OUTPUT_DIR)
required=['best_model.keras','class_names.json','training_metadata.json','classification_report.txt','confusion_matrix.png']
missing=[name for name in required if not (out/name).exists()]
if missing: raise FileNotFoundError(f'Thiếu artifacts: {missing}')
archive=shutil.make_archive('/content/plant_disease_training_artifacts','zip',OUTPUT_DIR)
destination='/content/drive/MyDrive/plant_disease_training_artifacts.zip'
shutil.copy2(archive,destination)
print('Đã lưu:',destination)
print('Tải ZIP từ Drive, giải nén và chép best_model.keras + class_names.json vào thư mục model/ của repository.')

## Sau khi huấn luyện

1. Kiểm tra training_metadata.json, classification_report.txt, confusion_matrix.png và learning_curves.png.
2. Chỉ dùng Accuracy/F1-score trong báo cáo sau khi notebook chạy hoàn tất và có kết quả thật.
3. Tải plant_disease_training_artifacts.zip từ Google Drive.
4. Giải nén, chép best_model.keras và class_names.json vào thư mục model/ của ứng dụng.
5. Chạy Streamlit. Ứng dụng ưu tiên model .keras và đọc thứ tự lớp từ JSON.

Tập test được giữ riêng khỏi quá trình cập nhật trọng số. Kết quả trên PlantVillage không đại diện chắc chắn cho ảnh lá ngoài đồng.